# PAISA OGDC model training

This Colab-compatible notebook runs the repository's reusable, versioned model trainer. It trains M-001 through M-004 on identical chronological splits, displays saved evaluation evidence, and exposes a ZIP of the immutable run. Model, target, feature, split, evaluation, and artifact logic stays in `src/paisa/experiments/`.

PAISA output is an academic experiment and is not financial advice.

## 1. Clone or use PAISA and install dependencies

The default Git ref is `main`. To exercise an open feature branch in Colab, set the `PAISA_GIT_REF` environment variable before running this cell.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Arfa01/PAISA.git"
GIT_REF = os.environ.get("PAISA_GIT_REF", "main")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    REPO_DIR = Path("/content/PAISA")
else:
    current = Path.cwd()
    REPO_DIR = current if (current / "pyproject.toml").exists() else current / "PAISA"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / "pyproject.toml").exists():
    raise RuntimeError(f"Existing path is not a PAISA checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
print("Repository:", REPO_DIR)
print("Git ref requested:", GIT_REF)
print("Python:", sys.version)

## 2. Train the shared CPU model variants

By default the reusable trainer fetches OGDC through `psx-dps`. If a real cleaned price file is already available, set `PAISA_PRICES_PATH` to its path before running this cell. No sample data is silently substituted for a failed real-data request.

In [ ]:
from paisa.experiments.trainer import run_training_stage

prices_value = os.environ.get("PAISA_PRICES_PATH")
prices_path = Path(prices_value) if prices_value else None
summary = run_training_stage(
    symbol="OGDC",
    start="2021-01-01",
    provider="psx-dps",
    output_dir=Path("artifacts/model_training"),
    random_state=42,
    prices_path=prices_path,
    create_zip=True,
)
summary

## 3. Inspect validation selection and saved metrics

In [ ]:
import json
import pandas as pd
from IPython.display import display

run_dir = Path(summary["artifact_directory"])
print("Run ID:", summary["run_id"])
print("Run directory:", run_dir)
print("Provisional champion (validation only):")
print(json.dumps(summary["provisional_champion"], indent=2))

for report_name in ("validation_leaderboard.csv", "test_leaderboard.csv", "baselines.csv"):
    report_path = run_dir / "reports" / report_name
    if report_path.exists():
        print(f"\n{report_name}")
        display(pd.read_csv(report_path))

## 4. Plot saved validation and test leaderboards

The trainer persists tabular evidence, not plot files. These charts are presentation-only views of the saved leaderboards; model selection remains validation-only.

In [ ]:
import matplotlib.pyplot as plt

validation_plot = pd.read_csv(run_dir / "reports" / "validation_leaderboard.csv").assign(split="validation")
test_plot = pd.read_csv(run_dir / "reports" / "test_leaderboard.csv").assign(split="test")
comparison = pd.concat([validation_plot, test_plot], ignore_index=True)
metric_names = [name for name in ("accuracy", "balanced_accuracy", "f1", "roc_auc") if name in comparison.columns]
fig, axes = plt.subplots(1, len(metric_names), figsize=(5 * len(metric_names), 4), squeeze=False)
for axis, metric_name in zip(axes[0], metric_names):
    comparison.pivot(index="model_id", columns="split", values=metric_name).plot(kind="bar", ax=axis)
    axis.set_title(metric_name.replace("_", " ").title())
    axis.set_xlabel("Registered model ID")
    axis.set_ylabel(metric_name)
    axis.tick_params(axis="x", rotation=45)
fig.suptitle("OGDC model comparison (test shown only after validation selection was frozen)")
fig.tight_layout()
plt.show()

## 5. Verify the backtesting prediction contract

The paths below can be supplied directly to `scripts/run_backtest.py` with a matching experiment configuration and daily OGDC price file.

In [ ]:
required_prediction_columns = {
    "schema_version", "model_id", "ticker", "feature_date",
    "target_date", "predicted_class", "probability_up",
    "probability_down", "actual_class",
}
prediction_paths = summary["prediction_paths"]
path_values = prediction_paths.values() if isinstance(prediction_paths, dict) else prediction_paths
for value in path_values:
    path = Path(value)
    columns = set(pd.read_csv(path, nrows=1).columns)
    missing = required_prediction_columns - columns
    if missing:
        raise AssertionError(f"{path} is missing prediction fields: {sorted(missing)}")
    print("Backtest-ready:", path)

## 6. Download the immutable run ZIP

In [ ]:
zip_path = Path(summary["zip_path"])
print("ZIP:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))